# Exploring the stochastic world of SGD - Visualization

In this notebook, we visualize the stochastic dynamics underlying **Stochastic Gradient Descent (SGD)** and compare its behavior with its deterministic counterpart, **Gradient Descent (GD)**.

To achieve this, we study the trajectories generated by the optimization algorithms on several low-dimensional **toy landscapes**. These visualizations help provide an intuitive understanding of the stochastic effects induced by random mini-batch sampling and how they influence the optimization process.

In addition, we investigate some relevant stochastic properties of SGD through empirical simulations and graphical analysis.

This notebook is divided into three main parts:

1. **Toy Landscapes**

   Definition and discussion of the optimization landscapes used throughout the notebook.

2. **Visualization Tools and Simulations**

   Implementation of the visualization toolbox and analysis of SGD and GD trajectories.

3. **Stochastic Properties**

   Empirical computation and discussion of relevant stochastic characteristics of SGD.


## General toolbox for Visualizing SGD Dynamics

In this section we will introduce the main visual toolbox we will use for this part.

To better understand how the discrete, noisy updates of SGD approximate a smooth continuous optimization path, this animation simulates a simple linear function fitting problem and tracks the optimization behavior simultaneously across four distinct viewpoints.

The target function is just $y = \frac{3}{2}x + 2$.

The animation is divided in 4 main panels:

* **Panel 1: 3D Parameter Space Landscape**
  Visualizes the full loss surface. It treats the parameters (weight $w$ and bias $b$) as positions on a physical bowl, showing the smooth descent of GD alongside the rugged, jittery path of SGD.
  
* **Panel 2: Step Vectors Compass Radar (Max-Normalized)**
  Isolates the driving forces acting on the model by mapping the true gradient and the minibatch gradient onto a unified coordinate origin $(0,0)$. By applying max-normalization, this panel magnifies microscopic variations at later steps, illustrating how SGD noise persists even after the true gradient decays to zero.

* **Panel 3: 2D Contour Overview**
  Provides a clean, top-down geometric look at the landscape's rings. This panel preserves the original spatial vectors, making it easy to see exactly when and where a minibatch calculation pulls the model off-course.

* **Panel 4: Data Space Model Fitting**
  Bridges the parameter space back to the input space. It plots the fixed data points, highlights the active minibatch points in orange, and shows how the model line pivots and slides live.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

In [2]:
# Generate a dataset of multiple points
# This is the Data Space
# The target function is y = 1.5x + 2
# We add some noise to the points
num_points = 40
x_data = np.linspace(-1, 3, num_points)
y_data = 1.5 * x_data + 2.0 + np.random.normal(0, 1, num_points)

In [3]:
# Define Loss and Gradient
def calculate_total_loss(w, b):
    predictions = w * x_data + b
    return np.mean((predictions - y_data) ** 2)

def get_gradient(w, b, x_sub, y_sub):
    predictions = w * x_sub + b
    errors = predictions - y_sub
    grad_w = (2 / len(x_sub)) * np.sum(x_sub * errors)
    grad_b = (2 / len(x_sub)) * np.sum(errors)
    return grad_w, grad_b

In [4]:
# Setup Optimization Paths
w_start, b_start = -0.5, -1.0  
lr = 0.1
steps = 30

In [5]:
# Run Standard Gradient Descent (Full Batch GD)
w_gd, b_gd = w_start, b_start
history_w_gd = [w_gd]
history_b_gd = [b_gd]
history_loss_gd = [calculate_total_loss(w_gd, b_gd)]

for _ in range(steps):
    fg_w, fg_b = get_gradient(w_gd, b_gd, x_data, y_data)
    w_gd = w_gd - lr * fg_w
    b_gd = b_gd - lr * fg_b
    history_w_gd.append(w_gd)
    history_b_gd.append(b_gd)
    history_loss_gd.append(calculate_total_loss(w_gd, b_gd))

In [6]:
# Run Stochastic Gradient Descent (Minibatch SGD)
w_sgd, b_sgd = w_start, b_start
batch_size = 4

history_w_sgd = [w_sgd]   
history_b_sgd = [b_sgd]
history_loss_sgd = [calculate_total_loss(w_sgd, b_sgd)]
history_minibatch_idx = [] 
history_full_grad = []     
history_mini_grad = []     

for _ in range(steps):
    fg_w, fg_b = get_gradient(w_sgd, b_sgd, x_data, y_data)
    history_full_grad.append((fg_w, fg_b))
    
    indices = np.random.choice(num_points, batch_size, replace=False)
    history_minibatch_idx.append(indices)
    
    mg_w, mg_b = get_gradient(w_sgd, b_sgd, x_data[indices], y_data[indices])
    history_mini_grad.append((mg_w, mg_b))
    
    w_sgd = w_sgd - lr * mg_w
    b_sgd = b_sgd - lr * mg_b
    history_w_sgd.append(w_sgd)
    history_b_sgd.append(b_sgd)
    history_loss_sgd.append(calculate_total_loss(w_sgd, b_sgd))

In [ ]:
# Set up 2x2 Plotting Grid
fig = plt.figure(figsize=(15, 12))
ax1 = fig.add_subplot(2, 2, 1, projection='3d') # 3D Landscape
ax2 = fig.add_subplot(2, 2, 2)                  # Max-Normalized Radar Compass
ax3 = fig.add_subplot(2, 2, 3)                  # 2D Contour (with kept vectors)
ax4 = fig.add_subplot(2, 2, 4)                  # Data Space

# Grid Setup
w_vals = np.linspace(-1, 3.5, 40)
b_vals = np.linspace(-1.5, 4.5, 40)
W, B = np.meshgrid(w_vals, b_vals)
Z = np.array([[calculate_total_loss(w_val, b_val) for w_val in w_vals] for b_val in b_vals]) # This is the loss landscape

# Panel 1: 3D Landscape
ax1.plot_surface(W, B, Z, cmap='viridis', alpha=0.4, edgecolor='none')
gd_path_3d, = ax1.plot([], [], [], 'g-', lw=2.5, label='GD Path')
sgd_path_3d, = ax1.plot([], [], [], 'r-', lw=2, label='SGD Path')
gd_dot_3d, = ax1.plot([], [], [], 'go', markersize=8)
sgd_dot_3d, = ax1.plot([], [], [], 'ro', markersize=8)
ax1.set_title("1. 3D Parameter Space Landscape")
ax1.set_xlabel("w")
ax1.set_ylabel("b")
ax1.legend()

# Panel 2: Max-Normalized Compass Radar Plot (Max Radius = 1.0)
for r in [0.25, 0.5, 0.75, 1.0]:
    circle = plt.Circle((0, 0), r, color='gray', fill=False, linestyle=':', alpha=0.5)
    ax2.add_patch(circle)

radar_full_arrow = ax2.quiver(0, 0, 0, 0, color='green', angles='xy', scale_units='xy', scale=1, width=0.012, label='True GD Vector Step')
radar_mini_arrow = ax2.quiver(0, 0, 0, 0, color='orange', angles='xy', scale_units='xy', scale=1, width=0.012, label='Minibatch SGD Vector Step')

ax2.set_xlim(-1.1, 1.1)
ax2.set_ylim(-1.1, 1.1)
ax2.axhline(0, color='black', lw=0.5, alpha=0.5)
ax2.axvline(0, color='black', lw=0.5, alpha=0.5)
ax2.set_title("2. Step Vectors Radar (Max-Normalized, Max Radius = 1)")
ax2.set_xlabel("Normalized Δw")
ax2.set_ylabel("Normalized Δb")
ax2.set_aspect('equal')
ax2.legend(loc='upper left')
ax2.grid(True, linestyle='--', alpha=0.3)

# Panel 3: 2D Contour (with retained spatial vectors)
contours = ax3.contour(W, B, Z, levels=20, cmap='viridis', alpha=0.3)
ax3.clabel(contours, inline=True, fontsize=8)
gd_path_2d, = ax3.plot([], [], 'g-', lw=2, alpha=0.7)
sgd_path_2d, = ax3.plot([], [], 'r-', lw=1.5, alpha=0.7)
gd_dot_2d, = ax3.plot([], [], 'go')
sgd_dot_2d, = ax3.plot([], [], 'ro')

# Arrow vectors on the contour landscape
landscape_full_arrow = ax3.quiver([], [], [], [], color='green', scale=10, width=0.008, label='Full Gradient')
landscape_mini_arrow = ax3.quiver([], [], [], [], color='orange', scale=10, width=0.008, label='Minibatch Gradient')

ax3.set_title("3. 2D Contour Overview with Landscape Vectors")
ax3.set_xlabel("w")
ax3.set_ylabel("b")
ax3.legend(loc='upper left')
ax3.grid(True)

# Panel 4: Data Space
ax4.scatter(x_data, y_data, color='blue', s=40, alpha=0.3, label='Dataset')
batch_points_scatter = ax4.scatter([], [], color='orange', s=100, edgecolor='black', zorder=4, label='Minibatch')
gd_model_line, = ax4.plot([], [], color='green', lw=2, label='GD Line')
sgd_model_line, = ax4.plot([], [], color='red', lw=2, label='SGD Line')
ax4.set_xlim(-1.5, 3.5)
ax4.set_ylim(-3, 8)
ax4.set_title("4. Data Space Model Fitting")
ax4.legend()
ax4.grid(True)

In [ ]:
# Animation Update Function
def update(frame):
    # Update 3D Landscape (Panel 1)
    gd_path_3d.set_data(history_w_gd[:frame+1], history_b_gd[:frame+1])
    gd_path_3d.set_3d_properties(history_loss_gd[:frame+1])
    sgd_path_3d.set_data(history_w_sgd[:frame+1], history_b_sgd[:frame+1])
    sgd_path_3d.set_3d_properties(history_loss_sgd[:frame+1])
    gd_dot_3d.set_data([history_w_gd[frame]], [history_b_gd[frame]])
    gd_dot_3d.set_3d_properties([history_loss_gd[frame]])
    sgd_dot_3d.set_data([history_w_sgd[frame]], [history_b_sgd[frame]])
    sgd_dot_3d.set_3d_properties([history_loss_sgd[frame]])
    
    # Update 2D Contour Paths (Panel 3)
    gd_path_2d.set_data(history_w_gd[:frame+1], history_b_gd[:frame+1])
    sgd_path_2d.set_data(history_w_sgd[:frame+1], history_b_sgd[:frame+1])
    gd_dot_2d.set_data([history_w_gd[frame]], [history_b_gd[frame]])
    sgd_dot_2d.set_data([history_w_sgd[frame]], [history_b_sgd[frame]])
    
    # Update Model Lines (Panel 4)
    x_vals = np.array([-2, 4])
    gd_model_line.set_data(x_vals, history_w_gd[frame] * x_vals + history_b_gd[frame])
    sgd_model_line.set_data(x_vals, history_w_sgd[frame] * x_vals + history_b_sgd[frame])
    
    # Update Vectors and Radar
    if frame < steps:
        cw_sgd, cb_sgd = history_w_sgd[frame], history_b_sgd[frame]
        fg_w, fg_b = history_full_grad[frame]
        mg_w, mg_b = history_mini_grad[frame]
        
        # Real spatial step updates (-lr * gradient)
        dx_full, dy_full = -fg_w * lr, -fg_b * lr
        dx_mini, dy_mini = -mg_w * lr, -mg_b * lr
        
        # Update Panel 3: Spatial Vectors on Landscape
        landscape_full_arrow.set_offsets([cw_sgd, cb_sgd])
        landscape_full_arrow.set_UVC(dx_full, dy_full)
        landscape_mini_arrow.set_offsets([cw_sgd, cb_sgd])
        landscape_mini_arrow.set_UVC(dx_mini, dy_mini)
        
        # Update Panel 2: Max-Normalized Compass Radar
        # Find magnitudes
        mag_full = np.sqrt(dx_full**2 + dy_full**2)
        mag_mini = np.sqrt(dx_mini**2 + dy_mini**2)
        max_mag = max(mag_full, mag_mini)
        
        if max_mag > 0:
            # Scale both down so that the largest vector has a length of exactly 1.0
            radar_dx_full = dx_full / max_mag
            radar_dy_full = dy_full / max_mag
            radar_dx_mini = dx_mini / max_mag
            radar_dy_mini = dy_mini / max_mag
        else:
            radar_dx_full, radar_dy_full, radar_dx_mini, radar_dy_mini = 0, 0, 0, 0
            
        radar_full_arrow.set_UVC(radar_dx_full, radar_dy_full)
        radar_mini_arrow.set_UVC(radar_dx_mini, radar_dy_mini)
        
        # Update highlighted minibatch points (Panel 4)
        batch_indices = history_minibatch_idx[frame]
        batch_points_scatter.set_offsets(np.column_stack((x_data[batch_indices], y_data[batch_indices])))
    else:
        # Clear dynamic elements on the last frame
        landscape_full_arrow.set_offsets([[0,0]]); landscape_full_arrow.set_UVC(0,0)
        landscape_mini_arrow.set_offsets([[0,0]]); landscape_mini_arrow.set_UVC(0,0)
        radar_full_arrow.set_UVC(0,0)
        radar_mini_arrow.set_UVC(0,0)
        batch_points_scatter.set_offsets(np.empty((0, 2)))

    return (gd_path_3d, sgd_path_3d, gd_dot_3d, sgd_dot_3d, radar_full_arrow, 
            radar_mini_arrow, gd_path_2d, sgd_path_2d, gd_dot_2d, sgd_dot_2d, 
            landscape_full_arrow, landscape_mini_arrow, batch_points_scatter, 
            gd_model_line, sgd_model_line)

# Play Animation
ani = FuncAnimation(fig, update, frames=len(history_w_sgd), interval=700, repeat=True)
plt.tight_layout()
plt.close()

HTML(ani.to_jshtml())

# Toy Landscapes

...

# Visualization

...

# Stochastic Properties

...